# Whisper Iterative Adapter Training — Bengali Long-Form ASR

Fine-tuning OpenAI Whisper on large Bengali speech datasets using **sequential LoRA adapter stacking** to avoid catastrophic forgetting.

---

## Strategy

Each training run covers one slice of the full dataset. Previous adapters are frozen before a new one is attached, so earlier knowledge is never overwritten.

| Iteration | Trainable | Frozen |
|-----------|-----------|--------|
| 1 | Adapter 1 | — |
| 2 | Adapter 2 | Adapter 1 |
| 3 | Adapter 3 | Adapters 1 & 2 |
| … | Adapter N | All previous |

At inference, all adapters are active together.

---

## Before You Run

Set these three variables in the **Configuration** cell below:

```python
IS_FIRST_TRAINING   = True   # True for iteration 1, False for 2+
CURRENT_ITERATION   = 1      # Which iteration this is (1, 2, 3 …)
PREVIOUS_ADAPTER_PATH = ""   # Path to last saved checkpoint (iterations 2+ only)
```

## 1 · Dependencies

In [ ]:
!pip uninstall -y torchcodec
!pip install -q torchaudio soundfile librosa transformers datasets accelerate
!pip install -q peft bitsandbytes evaluate jiwer webrtcvad audiomentations

## 2 · Configuration

Edit this cell before each run.

In [ ]:
# ── Iteration control ────────────────────────────────────────────────────────
IS_FIRST_TRAINING     = True
CURRENT_ITERATION     = 1
NUM_ITERATIONS        = 4
PREVIOUS_ADAPTER_PATH = "/kaggle/input/whisper-first-half-data-first-time-output/whisper-bengali-advanced/checkpoint-100"

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "/kaggle/input/datasets/tugstugi/bengali-ai-asr-submission/bengali-whisper-medium"

# ── Model ─────────────────────────────────────────────────────────────────────
LANGUAGE = "bengali"
TASK     = "transcribe"

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Audio ─────────────────────────────────────────────────────────────────────
SAMPLE_RATE    = 16000
CHUNK_LENGTH_S = 20.0   # seconds per chunk
OVERLAP_S      = 2.0    # overlap between chunks

# ── Labels ────────────────────────────────────────────────────────────────────
MAX_LABEL_LENGTH = 444
MIN_LABEL_LENGTH = 4

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE                 = 4
GRADIENT_ACCUMULATION_STEPS = 8   # effective batch = 32
LEARNING_RATE              = 1e-4
NUM_EPOCHS                 = 5
WARMUP_RATIO               = 0.1

USE_FP16                   = True
USE_GRADIENT_CHECKPOINTING = True

# ── Augmentation ──────────────────────────────────────────────────────────────
USE_AUGMENTATION   = True
AUGMENTATION_PROB  = 0.3

SEED = 42

print("=" * 60)
print(f"Mode      : {'FIRST TRAINING' if IS_FIRST_TRAINING else 'ITERATIVE'}")
print(f"Iteration : {CURRENT_ITERATION} / {NUM_ITERATIONS}")
print(f"Batch     : {BATCH_SIZE} × {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} effective")
print(f"LoRA      : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print("=" * 60)

## 3 · Imports

In [ ]:
import os, gc, warnings, re
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)
from peft import PeftModel, prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import Dataset, IterableDataset, Features, Array2D, Sequence, Value
import evaluate

warnings.filterwarnings("ignore")
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4 · Data Loading

In [ ]:
BASE_DIR           = Path("/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription")
TRAIN_AUDIO_DIR    = BASE_DIR / "train" / "audio"
TRAIN_ANNOTATION_DIR = BASE_DIR / "train" / "annotation"

if not TRAIN_AUDIO_DIR.exists():
    raise FileNotFoundError(f"Training audio directory not found: {TRAIN_AUDIO_DIR}")

def create_dataset_df(audio_dir, annotation_dir):
    data = []
    for audio_path in tqdm(sorted(audio_dir.glob("*.wav")), desc="Loading"):
        ann_path = annotation_dir / f"{audio_path.stem}.txt"
        if ann_path.exists():
            transcript = ann_path.read_text(encoding="utf-8").strip()
            if transcript:
                data.append({"id": audio_path.stem, "audio_path": str(audio_path), "transcript": transcript})
    return pd.DataFrame(data)

df_train = create_dataset_df(TRAIN_AUDIO_DIR, TRAIN_ANNOTATION_DIR)
print(f"Loaded {len(df_train)} samples")

## 5 · Text Preprocessing

Strip everything that is not Bengali script — punctuation, digits, English letters, and foreign scripts (Devanagari, Urdu/Arabic, Telugu, Malayalam).

In [ ]:
import string

_REMOVE = set(
    list(string.punctuation) +
    list("।—''|") +
    list(string.ascii_letters) +
    list("0123456789") +
    # Devanagari (Hindi)
    list("ानरेहमीोआक्तयलवचबपअखजंगदूणओएईधछघडैटभथँझऔऊळौठॉसुिश") +
    # Urdu / Arabic
    list("لاہیکنروتمےھدپسگجبشںئزضٹڈقڑچ") +
    # Telugu
    list("ఆానేరలస్") +
    # Malayalam
    list("ആസിചരാതലൂഗപർബയകങഓംേന") +
    # Zero-width & misc
    ["‍", "৳"]
)

def clean_transcript(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("\n", " ")
    text = "".join(c for c in text if c not in _REMOVE)
    return " ".join(text.split())

df_train["transcript"] = df_train["transcript"].apply(clean_transcript)
df_train = df_train[df_train["transcript"].str.len() > 0].reset_index(drop=True)

all_chars = Counter("".join(df_train["transcript"]))
print(f"Samples after cleaning : {len(df_train)}")
print(f"Unique characters      : {len(all_chars)}")
print(f"Top 10                 : {all_chars.most_common(10)}")

## 6 · Audio Statistics & Quality Filtering

In [ ]:
def get_duration(path):
    try:
        return sf.info(path).duration
    except Exception:
        return None

df_train["duration"] = [get_duration(p) for p in tqdm(df_train["audio_path"], desc="Durations")]
df_train = df_train.dropna(subset=["duration"])

df_train["word_count"] = df_train["transcript"].str.split().str.len()
df_train["char_count"] = df_train["transcript"].str.len()

before = len(df_train)
df_clean = df_train[df_train["duration"] >= 5.0].copy().reset_index(drop=True)

print(f"Removed (< 5 s) : {before - len(df_clean)}")
print(f"Remaining       : {len(df_clean)}")
print(f"Duration — mean {df_clean['duration'].mean():.1f}s  |  max {df_clean['duration'].max():.1f}s")
print(f"Words    — mean {df_clean['word_count'].mean():.0f}  |  max {df_clean['word_count'].max():.0f}")

## 7 · Data Portioning

The full dataset is divided into equal slices — one slice per training iteration. Only the current slice is used this run.

In [ ]:
PORTION_SIZE = len(df_clean) // NUM_ITERATIONS
start_idx    = (CURRENT_ITERATION - 1) * PORTION_SIZE
end_idx      = start_idx + PORTION_SIZE if CURRENT_ITERATION < NUM_ITERATIONS else len(df_clean)

df_portion = df_clean.iloc[start_idx:end_idx].copy()
train_df, val_df = train_test_split(df_portion, test_size=0.1, random_state=SEED)

print(f"Total dataset  : {len(df_clean)}")
print(f"This slice     : samples {start_idx}–{end_idx}  ({len(df_portion)} total)")
print(f"Train / Val    : {len(train_df)} / {len(val_df)}")

## 8 · Audio Preprocessing

Pipeline applied to every sample:

1. **RMS normalisation** → −20 dBFS  
2. **Robust resampling** → 16 kHz → 8 kHz → 16 kHz (smooths codec artefacts)  
3. **Waveform augmentation** → time-stretch, pitch-shift, Gaussian noise  
4. **SpecAugment** → time & frequency masking on the mel spectrogram

In [ ]:
def rms_normalize(audio, target_db=-20.0):
    rms = np.sqrt(np.mean(audio ** 2))
    if rms == 0:
        return audio
    gain = 10 ** (target_db / 20.0) / rms
    return np.clip(audio * gain, -1.0, 1.0)


def robust_resample(audio, sr=16000):
    audio_8k = librosa.resample(audio, orig_sr=sr, target_sr=8000)
    return librosa.resample(audio_8k, orig_sr=8000, target_sr=16000)


def augment_waveform(audio, sr=16000):
    if not USE_AUGMENTATION or np.random.random() > AUGMENTATION_PROB:
        return audio
    aug = audio.copy()
    if np.random.random() < 0.5:
        rate = 0.95 + np.random.random() * 0.1
        aug = librosa.effects.time_stretch(aug, rate=rate)
        aug = aug[:len(audio)] if len(aug) > len(audio) else np.pad(aug, (0, len(audio) - len(aug)))
    if np.random.random() < 0.5:
        n_steps = np.random.randint(-2, 3)
        aug = librosa.effects.pitch_shift(aug, sr=sr, n_steps=n_steps)
    if np.random.random() < 0.3:
        noise = np.random.normal(0, 0.002 + np.random.random() * 0.003, len(aug))
        aug = np.clip(aug + noise, -1.0, 1.0)
    return aug


def specaugment(features, time_mask=15, freq_mask=10):
    if not USE_AUGMENTATION or np.random.random() > AUGMENTATION_PROB:
        return features
    f = features.copy()
    n_mels, n_frames = f.shape
    for _ in range(2):
        t  = np.random.randint(1, min(time_mask + 1, n_frames))
        t0 = np.random.randint(0, max(1, n_frames - t))
        f[:, t0:t0 + t] = 0
    for _ in range(2):
        fm  = np.random.randint(1, min(freq_mask + 1, n_mels))
        f0  = np.random.randint(0, max(1, n_mels - fm))
        f[f0:f0 + fm, :] = 0
    return f

print("Preprocessing functions ready.")

## 9 · Audio Chunking with Word Alignment

In [ ]:
def chunk_audio(audio, sr=16000):
    chunk_samples   = int(CHUNK_LENGTH_S * sr)
    overlap_samples = int(OVERLAP_S * sr)
    stride          = chunk_samples - overlap_samples
    chunks, start   = [], 0
    while start < len(audio):
        end   = min(start + chunk_samples, len(audio))
        chunk = audio[start:end]
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
        chunks.append({"audio": chunk, "start_time": start / sr, "end_time": end / sr})
        if end >= len(audio):
            break
        start += stride
    return chunks


def process_sample(audio_path, transcript, processor):
    try:
        audio, sr    = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        audio        = rms_normalize(audio)
        audio        = robust_resample(audio, SAMPLE_RATE)
        audio        = augment_waveform(audio, SAMPLE_RATE)
        total_dur    = len(audio) / sr
        words        = transcript.split()
        wps          = len(words) / total_dur if total_dur > 0 else 0

        def make_item(chunk_audio, chunk_text):
            feats  = processor.feature_extractor(chunk_audio, sampling_rate=SAMPLE_RATE).input_features[0]
            feats  = specaugment(feats)
            labels = processor.tokenizer(chunk_text).input_ids
            return feats, labels

        def valid(labels):
            return MIN_LABEL_LENGTH <= len(labels) <= MAX_LABEL_LENGTH

        if total_dur <= CHUNK_LENGTH_S:
            feats, labels = make_item(audio, transcript)
            return [{"input_features": feats, "labels": labels}] if valid(labels) else []

        results = []
        for c in chunk_audio(audio, sr):
            s_idx      = int(c["start_time"] * wps)
            e_idx      = min(s_idx + int((c["end_time"] - c["start_time"]) * wps), len(words))
            chunk_text = " ".join(words[s_idx:e_idx])
            if not chunk_text:
                continue
            feats, labels = make_item(c["audio"], chunk_text)
            if len(labels) > MAX_LABEL_LENGTH:
                truncated = []
                for w in words[s_idx:e_idx]:
                    test = " ".join(truncated + [w])
                    if len(processor.tokenizer(test).input_ids) <= MAX_LABEL_LENGTH:
                        truncated.append(w)
                    else:
                        break
                if not truncated:
                    continue
                feats, labels = make_item(c["audio"], " ".join(truncated))
            if valid(labels):
                results.append({"input_features": feats, "labels": labels})
        return results

    except Exception as e:
        warnings.warn(f"Skipping {audio_path}: {e}")
        return []

print("Chunking functions ready.")

## 10 · Model Loading

**Iteration 1** — base Whisper + one new LoRA adapter.  
**Iterations 2 +** — base Whisper + previous adapter (frozen) + one new LoRA adapter.

Freezing previous adapters is the mechanism that prevents catastrophic forgetting.

In [ ]:
print("Loading tokenizer & processor …")
tok_path = BASE_MODEL if IS_FIRST_TRAINING else PREVIOUS_ADAPTER_PATH
tokenizer = WhisperTokenizer.from_pretrained(tok_path, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(tok_path, language=LANGUAGE, task=TASK)
print(f"Vocabulary size: {len(tokenizer)}")

LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    modules_to_save=["embed_tokens", "lm_head"],
)

if IS_FIRST_TRAINING:
    print("\nIteration 1 — loading base model + Adapter 1")
    model = WhisperForConditionalGeneration.from_pretrained(
        BASE_MODEL, device_map="auto", torch_dtype=torch.float16
    )
    if len(tokenizer) != model.config.vocab_size:
        model.resize_token_embeddings(len(tokenizer))
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print("Adapter 1 attached.")

else:
    print(f"\nIteration {CURRENT_ITERATION} — stacking adapters (previous frozen)")
    base = WhisperForConditionalGeneration.from_pretrained(
        BASE_MODEL, device_map="auto", torch_dtype=torch.float16
    )
    if len(tokenizer) != base.config.vocab_size:
        base.resize_token_embeddings(len(tokenizer))

    model = PeftModel.from_pretrained(base, PREVIOUS_ADAPTER_PATH, adapter_name="adapter_prev")
    model.set_adapter("adapter_prev")

    frozen = sum(1 for n, p in model.named_parameters()
                 if ("lora" in n.lower() or "adapter" in n.lower()) and not p.requires_grad_(False))
    print(f"Previous adapter frozen ({frozen} parameter groups).")

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    model.add_adapter("adapter_new", lora_config)
    model.set_adapter("adapter_new")
    print("New adapter attached and set as active.")

print()
model.print_trainable_parameters()

## 11 · Streaming Datasets

In [ ]:
def make_generator(df):
    def gen():
        for _, row in df.iterrows():
            for item in process_sample(row["audio_path"], row["transcript"], processor):
                yield item
    return gen

_features = Features({
    "input_features": Array2D(shape=(80, 3000), dtype="float32"),
    "labels": Sequence(Value("int64")),
})

train_dataset = IterableDataset.from_generator(make_generator(train_df), features=_features)
val_dataset   = IterableDataset.from_generator(make_generator(val_df),   features=_features)

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Streaming datasets ready.")

## 12 · Data Collator

In [ ]:
@dataclass
class SpeechCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch  = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = SpeechCollator(processor=processor)
print("Data collator ready.")

## 13 · Evaluation Metric (WER)

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

## 14 · Training Arguments

In [ ]:
ESTIMATED_CHUNKS = len(train_df) * 200
MAX_STEPS        = NUM_EPOCHS * (ESTIMATED_CHUNKS // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
WARMUP_STEPS     = int(WARMUP_RATIO * MAX_STEPS)

print(f"Estimated steps : {MAX_STEPS}  |  Warmup : {WARMUP_STEPS}")

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-bengali-advanced",

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",

    fp16=USE_FP16,
    max_grad_norm=1.0,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,

    logging_steps=max(1, MAX_STEPS // 20),
    eval_strategy="steps",
    eval_steps=max(1, MAX_STEPS // 10),
    save_steps=max(1, MAX_STEPS // 10),
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=MAX_LABEL_LENGTH,
    generation_num_beams=5,
    remove_unused_columns=False,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    load_best_model_at_end=False,
    metric_for_best_model="wer",
    greater_is_better=False,

    report_to=["tensorboard"],
    push_to_hub=False,
)

## 15 · Gradient Monitor Callback

In [ ]:
class GradientMonitor(TrainerCallback):
    def __init__(self):
        self.norms = []
        self.step  = 0

    def on_step_end(self, args, state, control, **kwargs):
        self.step += 1
        model = kwargs.get("model")
        total = sum(p.grad.data.norm(2).item() ** 2
                    for p in model.parameters() if p.grad is not None) ** 0.5
        self.norms.append(total)
        if self.step % 50 == 0:
            if total > 10:
                print(f"[step {self.step}] ⚠ high grad norm: {total:.2f}")
            elif total < 0.01:
                print(f"[step {self.step}] ⚠ very low grad norm: {total:.4f}")

    def on_train_end(self, args, state, control, **kwargs):
        if self.norms:
            print(f"Grad norm — mean {np.mean(self.norms):.4f}  max {np.max(self.norms):.4f}  std {np.std(self.norms):.4f}")

grad_monitor = GradientMonitor()

## 16 · PEFT Forward Patch

PEFT injects `input_ids` / `inputs_embeds` into the forward signature. Whisper rejects both. This patch intercepts them before they reach the model.

In [ ]:
import types

def _patched_forward(self, input_ids=None, inputs_embeds=None, **kwargs):
    return self.original_forward(**kwargs)

_target = model.base_model if hasattr(model, "base_model") else model
if not hasattr(_target, "original_forward"):
    _target.original_forward = _target.forward
_target.forward = types.MethodType(_patched_forward, _target)

print("Forward patch applied.")

## 17 · Training

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
    callbacks=[grad_monitor],
)

if torch.cuda.is_available():
    print(f"GPU memory before training: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

trainer.train()
print("Training complete.")

## 18 · Save Checkpoint

In [ ]:
OUTPUT_DIR = "./whisper-bengali-final"

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved to: {OUTPUT_DIR}")

if CURRENT_ITERATION < NUM_ITERATIONS:
    print(f"\nNext run — set:")
    print(f"  IS_FIRST_TRAINING     = False")
    print(f"  CURRENT_ITERATION     = {CURRENT_ITERATION + 1}")
    print(f"  PREVIOUS_ADAPTER_PATH = '{OUTPUT_DIR}/checkpoint-XXX'")
else:
    print("All iterations complete. Load all adapters together for inference.")

## 19 · Cleanup

In [ ]:
del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")